# 94 — Load, Test & Enhance: 93-Supplier_to_moving

Test notebook for `StrategyPipeline` from `93-Supplier_to_moving.py`.
Each test exposes `df_s`, `pa`, `safe`, and `action` for interactive inspection.

| Test | Setup |
|------|-------|
| Test 01 | 7 player-0 planets, no enemies — all Suppliers, expect no action |
| Test 02 | Same + enemy at (5, 60) — Conquerors emerge, Suppliers should reinforce |

In [ ]:
%run 93-Supplier_to_moving.py

In [ ]:
import copy, math, random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


def simulate_with_action(obs, action0, n_steps, current_step=0):
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, "+"+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

# Test function

## Test 01 — All player-0 planets, no enemies

Planets (all owner=0, angular_velocity=0.02 rad/step):
- `id=0` (300 ships, +4): x=35, y=5  ← moving (dist+r ≈ 49.8)
- `id=1` (90 ships, +2): x=25, y=5   ← fix
- `id=2` (200 ships, +4): x=15, y=15 ← fix
- `id=3` (60 ships, +1): x=5, y=25   ← fix
- `id=4` (300 ships, +5): x=5, y=35  ← fix (dist+r ≈ 50.0)
- `id=5` (50 ships, +3): x=30, y=30  ← moving (dist+r ≈ 30.4)
- `id=6` (30 ships, +2): x=20, y=50  ← moving (dist+r ≈ 31.7)

Expected: no enemy targets → all Suppliers → no action.

In [ ]:
R = lambda prod: 1 + math.log(prod)
obs01 = Obs(
    planets=[
        [0,  0, 35.0,  5.0, R(4), 300, 4],
        [1,  0, 25.0,  5.0, R(2),  90, 2],
        [2,  0, 15.0, 15.0, R(4), 200, 4],
        [3,  0,  5.0, 25.0, R(1),  60, 1],
        [4,  0,  5.0, 35.0, R(5), 300, 5],
        [5,  0, 30.0, 30.0, R(3),  50, 3],
        [6,  0, 20.0, 50.0, R(2),  30, 2],
    ],
    angular_velocity=0.02,
)
df_s01, pd01 = StrategyPipeline._01_get_obs_dataframe(obs01, step=0, num_agents=2)
df_s01

In [ ]:
pa01 = StrategyPipeline._02_get_all_opportunities(df_s01, pd01, player_id=0)
pa01

In [ ]:
safe01 = StrategyPipeline._03_filter_collision(pa01)
safe01

In [ ]:
action01 = StrategyPipeline._04_score_and_decide(safe01, player_id=0)
print("Action:", action01)
snaps01 = simulate_with_action(copy.deepcopy(obs01), action01, 30)
make_animation(snaps01, title='Test 01 — All own planets, no action', interval=200)

## Test 02 — Same + enemy at (5, 60)

Same 7 player-0 planets as Test 01, plus:
- `id=7` (owner=1, 20 ships, +2): x=5, y=60  ← moving enemy (dist+r ≈ 47.8)

Expected: planets near enemy (id=4, id=6) become Conquerors; farther ones become Suppliers and reinforce them.

In [ ]:
obs02 = Obs(
    planets=[
        [0,  0, 35.0,  5.0, R(4), 300, 4],
        [1,  0, 25.0,  5.0, R(2),  90, 2],
        [2,  0, 15.0, 15.0, R(4), 200, 4],
        [3,  0,  5.0, 25.0, R(1),  60, 1],
        [4,  0,  5.0, 35.0, R(5), 300, 5],
        [5,  0, 30.0, 30.0, R(3),  50, 3],
        [6,  0, 20.0, 50.0, R(2),  30, 2],
        [7,  1,  5.0, 60.0, R(2),  20, 2],
    ],
    angular_velocity=0.02,
)
df_s02, pd02 = StrategyPipeline._01_get_obs_dataframe(obs02, step=0, num_agents=2)
df_s02

In [ ]:
pa02 = StrategyPipeline._02_get_all_opportunities(df_s02, pd02, player_id=0)
pa02

In [ ]:
safe02 = StrategyPipeline._03_filter_collision(pa02)
safe02

In [ ]:
action02 = StrategyPipeline._04_score_and_decide(safe02, player_id=0)
print("Action:", action02)
snaps02 = simulate_with_action(copy.deepcopy(obs02), action02, 30)
make_animation(snaps02, title='Test 02 — Enemy at (5,60), Conquerors + Suppliers', interval=200)